In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)

feature_cols = ["eua_lag_1", "eua_lag_5", "spread_lag_1", "renewable_share", "HDD", "eua_vol_21d", "month"]
target_col = "eua_price"

split_idx = int(len(df) * 0.80)
train_df, test_df = df.iloc[:split_idx], df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

# Time series split validation loop
tscv = TimeSeriesSplit(n_splits=3)
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

# Train final model
final_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
final_model.fit(X_train, y_train)

# Save predictions
oos_preds = final_model.predict(X_test)
pd.DataFrame({"actual": y_test, "xgb_pred": oos_preds}, index=test_df.index).to_csv("data/processed/xgb_preds.csv")

# Plot & save feature importance
xgb.plot_importance(final_model, importance_type="gain")
plt.tight_layout()
plt.savefig("figures/feature_importance.png", dpi=150)
plt.show()